<style>
    .jp-RenderedHTMLCommon {
        font-family: "Times New Roman", Times, serif !important;
        font-size: 16px !important;
    }
</style>

### **1. Постановка задачи**

Рассмотрим набор различных точек на отрезке $[a, b]$: $x_0 < x_1 < \dots < x_n$, в
которых заданы значения функции $f(x)$ так, что $f_i = f(x), \quad і = 0,1, \dots, n$. Требуется востановить значение $f(x)$ и в других точках $х \in [a, b]$.

Вариант исходных данных:

Для построения таблицы $(х_i, f_i)$ дано:

$$[a, b] = [\alpha_j, 1 + \alpha_j]$$
где $\alpha_j = 0.1 + 0.05j$, где $j$ - порядковый номер в списке подгруппы.

$$x_i = \alpha_j + ih, \quad h = \frac{1}{n}, \quad i = 0,1,\dots,n$$
где $n = 10$.

$$f(x) = \alpha_j e^x + (1 - \alpha_j)\cos{x}$$

Точки восстановления: $x^* = x_0 + \frac{2}{3}h,\quad x^{**} = x_{n/2} + \frac{1}{2}h,\quad x^{***} = x_n - \frac{1}{3}h$.

Таким образом, для $j = 3$, $n = 10$ получим:

$$\alpha_j = 0.1 + 0.05 \cdot 3 =0.25$$

$$[a, b] = [0.25, 1.25]$$

$$h = 0.1 \qquad x_i = 0.25 + 0.1 \cdot i \qquad i = 0,1, \dots, 10$$

$$f(x) = 0.25 e^x + 0.75 \cos{x}$$

Точки восстановления: $x^* = 0.25 + \frac{1}{15} = \frac{19}{60}, \quad x^{**} = 0.25 + 0.05 = 0.75, \quad x^{***} = 1.25 - \frac{1}{30} = \frac{73}{60},$

где $x_0 = 0.25, \quad x_n = 1.25.$

Необходимо: 
1. Методом наименьших квадратов построить элемент наилучшего приближения $\Phi_0$ 5-й степени при $p(x) = 1$. 
2. Интерполяционный многочлен Лагранжа
3. Представление остатка интерполирования в форме Ньютона
4. Построение интерполяционного многочлена Лагранжа с измененной сеткой узлов при помощи многочлена Чебышева
5. Построение интерполяционного многочлена Ньютона внутри таблицы.

--- 
### **2. Методом наименьших квадратов построить элемент наилучшего приближения $\Phi_0$ 5-й степени при $p(x) = 1$.**

Искомый элемент $\Phi_0(x)$ представляется в виде линейной комбинации базисных функций $\varphi_i(x) = x^i$:
$$\Phi_0(x) = \sum_{i=0}^{5} c_i x^i$$

Коэффициенты $c_i$ определяются из условия ортогональности невязки подпространству:
$$(f - \Phi_0, \varphi_j) = 0, \quad j = 0,1,\dots,5$$

В случае таблично заданной функции скалярное произведение задаётся в дискретной форме:
$$(f,g) = \sum_{k=0}^{n} f(x_k) g(x_k)$$

Тогда коэффициенты $c_i$ находятся из системы уравнений:
$$\sum_{i=0}^{k} \left( \sum_{j=0}^{n} x_j^{i+k} \right) c_i = \sum_{j=0}^{n} f(x_j) x_j^k, \quad k = 0,1,\dots,5$$

Решение данной системы позволяет построить многочлен $\Phi_0(x)$. Для решения системы воспользуемся методом Гаусса.

In [1]:
import numpy as np
import linear_algebra as linalg
import pandas as pd

def f(x: float) -> float:
    return 0.25 * np.exp(x) + 0.75*np.cos(x)

n = 10
alpha = 0.25
h = 1/n
K = 5

X = [alpha + i*h for i in range(n+1)]
F_x = [float(f(xi)) for xi in X]
recovery_x = (X[0] + h * 2/3, X[int(n/2)] + h * 1/2, X[n] - h * 1/3)

df = pd.DataFrame({
    "x": X,
    "f(x)": F_x
}).set_index("x").T
df 

x,0.25,0.35,0.45,0.55,0.65,0.75,0.85,0.95,1.05,1.15,1.25
f(x),1.047691,1.059296,1.067413,1.072707,1.075948,1.078017,1.079899,1.08269,1.087591,1.095914,1.109078


In [2]:
def gaussian_method(a: list[list])->list[float]:
    def change_system_for_select_main_element(system, j):
        max_row = max(system[j:], key=lambda row: row[j])
        max_index = system.index(max_row)
        system[max_index], system[j] = system[j], system[max_index]
        m = 1 if max_index == 0 else -1
        return system, m
    def straight_stroke(a):
        n = len(a)
        for k in range(0,n):
            a, _ = change_system_for_select_main_element(a,k)
            for j in range(n,k-1,-1):
                a[k][j] /= a[k][k]
            for i in range(k+1,n):
                for j in range(n,k-1,-1):
                    a[i][j] = a[i][j] - a[i][k]*a[k][j]
        return a
    def reverse_stroke(a: list[list[float]])->list[float]:
        n = len(a)
        x = [0.] * n
        x[n-1] = a[n-1][n]
        for i in range(n-2,-1,-1):
            s = 0
            for j in range(i+1,n):
                s+=a[i][j]*x[j]
            x[i] = a[i][n] - s
        return x
    
    return reverse_stroke(straight_stroke(a))

b = np.zeros(K+1)
A = np.zeros((K+1, K+2)) #K+2 для расшриенной матрицы

for row in range(K+1):
    b[row] = sum(F_x[j] * (X[j]**row) for j in range(n+1))
    A[row][-1] = b[row]
    for col in range (K+1):
        A[row][col] = sum(X[t]**(col+row) for t in range(n+1))

print("\n", "Вектор b:")
linalg.show_matrix([b])
print("Расширенная матрица A|b:")
linalg.show_matrix(A)
c = gaussian_method(A.tolist())
print("Коэффициенты c:")
linalg.show_matrix([c], e=True)
print(c)


 Вектор b:
 11.8562   8.9460   7.9356   7.7712   8.0871   8.7548
Расширенная матрица A|b:
 11.0000   8.2500   7.2875   7.1156   7.3888   7.9852  11.8562
  8.2500   7.2875   7.1156   7.3888   7.9852   8.8716   8.9460
  7.2875   7.1156   7.3888   7.9852   8.8716  10.0566   7.9356
  7.1156   7.3888   7.9852   8.8716  10.0566  11.5751   7.7712
  7.3888   7.9852   8.8716  10.0566  11.5751  13.4829   8.0871
  7.9852   8.8716  10.0566  11.5751  13.4829  15.8560   8.7548
Коэффициенты c:
1.0000e+00 2.4987e-01 -2.4935e-01 4.0017e-02 4.4013e-02 2.3911e-04
[1.0000108564660362, 0.24986724682201125, -0.2493507662598533, 0.040017287413288805, 0.044013441319538604, 0.00023911200378093586]


Таким образом, мы получили коэффициенты $c_i$ многочлена наилучшего среднеквадратического приближения $\Phi_0$:
$$\Phi_0(x) = 1.0000108564660362 + 0.24986724682201125x + -0.2493507662598533x^2 + 0.040017287413288805x^3 +$$
$$ + 0.044013441319538604x^4 + 0.00023911200378093586x^5$$

Для оценки погрешности вычисления полинома вычислим:

$$\Delta f = \left(\sum_{j=0}^{n} (\Phi_0(x_j) - f(x_j))^2 \right)^{\frac{1}{2}}$$

И вычислим невязку в точках восстановления:
$$\left|r(x^*)\right| = \left|\Phi_0(x^*) - f (x^*)\right|$$

In [3]:
def Phi(x: float, coeffs: list[float]) -> float:
    return sum(coeffs[i]*x**i for i in range(len(coeffs)))

inaccuracy = np.sqrt(sum((f(x) - Phi(x, c))**2 for x in X))
print ("Погрешность вычислений:", inaccuracy)

residuals =  [abs(Phi(x, c) - f(x)) for x in recovery_x]

df = pd.DataFrame({
    "Точки восстановления": recovery_x,
    "f(x)": [f(x) for x in recovery_x],
    "Phi(x)": [Phi(x, coeffs=c) for x in recovery_x],
    "Невязка": residuals
})
df = df.style.format({"Невязка": "{:.5e}"})
df


Погрешность вычислений: 5.340821818398503e-08


,Точки восстановления,f(x),Phi(x),Невязка
0,0.316667,1.055845,1.055845,4.60907e-08
1,0.800000,1.078915,1.078915,8.32533e-09
2,1.216667,1.104060,1.104060,3.40238e-08


### **3. Интерполяционный многочлен Лагранжа**

Задача интерполирования заключается в построении функции $P_n(x)$, которая совпадает с заданной функцией $f(x)$ в узлах интерполирования $x_0, x_1, \dots, x_n$, то есть:
$$P_n(x_i) = f(x_i), \quad i = 0,1,\dots,n.$$

В качестве аппроксимирующей функции выбирается алгебраический многочлен степени не выше $n$:
$$P_n(x) = \sum_{k=0}^{n} a_k x^k.$$

Для построения интерполяционного многочлена используется формула Лагранжа, которая позволяет записать $P_n(x)$ в виде:
$$P_n(x) = \sum_{i=0}^{n} \frac{\omega_{n+1}(x)}{(x-x_i)\omega_{n+1}'(x_i)}f(x_i),$$
где $\quad \omega_{n+1}(x) = \prod_{k=0}^{n}(x - x_i) = (x-x_0)\dots(x-x_n).$

$$\omega'_{n+1}(x) = \sum_{m=0}^{n} \left( \prod_{\substack{k=0 \\ k \ne m}}^{n} (x - x_k) \right)$$

Если подставить $x_i$, получим:

$$\omega'_{n+1}(x_i) = \prod_{\substack{k=0 \\ k \ne i}}^{n} (x_i - x_k)$$

In [4]:
def omega(x: float, X: list[float]) -> float:
    return np.prod([x - xi for xi in X])

def omega_derivative(x: float, X: list[float]) -> float:
    total = 0.0
    for i in range(len(X)):
        total += np.prod([x - X[j] for j in range(len(X)) if j != i])
    return total

def lagrange_polynomial(x: float, X: list[float], F_x: list[float]) -> float:
    total = 0.0
    for i in range(len(X)):
        numerator = omega(x, X) * F_x[i]
        denominator = (x  - X[i])* omega_derivative(X[i], X)
        total += numerator / denominator 
    return total

Для оценки погрешности вычислим невязку в точках восстановления:
$$\left|r(x^*)\right| = \left|P_{n}(x^*) - f (x^*)\right|$$

In [5]:
residuals = [abs(lagrange_polynomial(x, X, F_x) - f(x)) for x in recovery_x]

df = pd.DataFrame({
    "Точки восстановления": recovery_x,
    "f(x)": [f(x) for x in recovery_x],
    "P_n(x)": [lagrange_polynomial(x, X, F_x) for x in recovery_x],
    "Невязка": residuals
})
df = df.style.format({"Невязка": "{:.5e}"})
df


,Точки восстановления,f(x),P_n(x),Невязка
0,0.316667,1.055845,1.055845,4.81837e-14
1,0.800000,1.078915,1.078915,1.11022e-15
2,1.216667,1.104060,1.104060,1.11910e-13


### **4. Представление остатка интерполирования в форме Ньютона**

Под остатком интерполирования понимается разность между исходной функцией и интерполяционным многочленом:

$$r_n(x) = f(x) - P_n(x).$$

Остаток интерполирования может быть представлен через разделённые разности:

$$f(x_i, x_j) = \frac{f(x_j) - f(x_i)}{x_j - x_i}\quad \text{- разделенная разность 1-го порядка}$$

$$f(x_i, x_j, x_k) = \frac{f(x_j, x_k) - f(x_i, x_j)}{x_k - x_i}\quad \text{- разделенная разность 2-го порядка}$$


$$f(x_0, x_1, \dots, x_{n+1}) = \frac{f(x_1, \dots, x_{n+1}) - f(x_0, \dots, x_n)}{x_{n+1} - x_0} \quad \text{- разделенная разность } (n+1)\text{-го порядка}$$

Тогда остаток интерполирования в форме Ньютона будет выглядеть следующим образом:

$$r_n(x) = \omega_{n+1}(x) f(x, x_0, \dots, x_n)$$

Для реализации аппарата разделенной разности воспользуемся динамическим программированием. Реализация через одномерный массив экономит память, обновляя значения на месте. Реализация через двумерный массив строит полную таблицу разностей для наглядности и удобства отладки.

In [6]:
def divided_difference_1d(X: list[float], F_x: list[float]) -> list[float]:
    """
    Вычисление разделенной разности f(x_i, ..., x_j), используя одномерный массив
    X: список узлов
    F_x: список значений функции в узлах
    """
    n = len(X)
    dp = F_x.copy()
   
    for i in range(1,n):
        for j in range(n-i):            
            dp[j] = (dp[j+1] - dp[j]) / (X[j+i] - X[j])
    return dp

def divided_difference_2d(X: list[float], F_x: list[float]) -> list[list[float]]:
    """
    Вычисление разделенной разности f(x_i, ..., x_j), используя двумерный массив
    X: список узлов
    F_x: список значений функции в узлах
    """
    n = len(X)
    dp = [[0.0] * n for _ in range(n)]
    dp[0] = F_x.copy()

    for i in range(1, n):
        for j in range(n - i):
            dp[i][j] = (dp[i-1][j + 1] - dp[i-1][j]) / (X[j+i] - X[j])
 
    return dp

def newton_remainder(x: float, X: list[float], F_x: list[float]) -> tuple[float, list[list[float]]]:
    table = divided_difference_2d([x] + X, [f(x)] + F_x)
    return table[len(X)][0] * omega(x, X), table

residuals = []
for x in recovery_x:
    r, table = newton_remainder(x, X, F_x)
    residuals.append(abs(r))

print("Остатки интерполирования в форме Ньютона для точек восстановления:")
df_r = pd.DataFrame({
    "Точки восстановления": recovery_x,
    "Остаток в форме Ньютона": residuals
})
display(df_r)
#print(divided_difference_1d([x] + X, [f(x)] + F_x)[0])
#print(table[n-1][0])

n = len(table)

df = pd.DataFrame(table).T
rows, cols = df.shape
for i in range(n):
    df.iloc[i, n-i:] = np.nan
df.index = ["x"] + [f"x{i}" for i in range(0, n-1)]
df.columns = [f"Порядок {i}" for i in range(n)]   
df.insert(0, "x", [recovery_x[-1]] + X)   
print("\nТаблица разделенных разностей для формы Ньютона:")
df = df.style.format({f"Порядок {i}": "{:.3e}" for i in range(n)}, na_rep="")
display(df)

Остатки интерполирования в форме Ньютона для точек восстановления:


,Точки восстановления,Остаток в форме Ньютона
0,0.316667,4.890902e-14
1,0.800000,1.351824e-15
2,1.216667,1.104955e-13



Таблица разделенных разностей для формы Ньютона:


,x,Порядок 0,Порядок 1,Порядок 2,Порядок 3,Порядок 4,Порядок 5,Порядок 6,Порядок 7,Порядок 8,Порядок 9,Порядок 10,Порядок 11
x,1.216667,1.104e+00,5.831e-02,-6.663e-02,1.406e-01,4.465e-02,3.511e-04,-2.159e-04,1.815e-04,2.669e-05,6.568e-08,-4.951e-09,2.695e-08
x0,0.250000,1.048e+00,1.161e-01,-1.744e-01,1.109e-01,4.446e-02,4.518e-04,-2.825e-04,1.744e-04,2.668e-05,6.601e-08,-4.053e-09,
x1,0.350000,1.059e+00,8.117e-02,-1.412e-01,1.286e-01,4.468e-02,2.823e-04,-1.604e-04,1.958e-04,2.674e-05,6.195e-08,,
x2,0.450000,1.067e+00,5.293e-02,-1.026e-01,1.465e-01,4.482e-02,1.861e-04,-2.336e-05,2.172e-04,2.679e-05,,,
x3,0.550000,1.073e+00,3.241e-02,-5.864e-02,1.644e-01,4.492e-02,1.721e-04,1.286e-04,2.386e-04,,,,
x4,0.650000,1.076e+00,2.069e-02,-9.309e-03,1.824e-01,4.500e-02,2.493e-04,2.957e-04,,,,,
x5,0.750000,1.078e+00,1.882e-02,4.541e-02,2.004e-01,4.513e-02,4.266e-04,,,,,,
x6,0.850000,1.080e+00,2.791e-02,1.055e-01,2.185e-01,4.534e-02,,,,,,,
x7,0.950000,1.083e+00,4.901e-02,1.711e-01,2.366e-01,,,,,,,,
x8,1.050000,1.088e+00,8.323e-02,2.420e-01,,,,,,,,,


### **5. Построение интерполяционного многочлена Лагранжа с измененной сеткой узлов при помощи многочлена Чебышева**

**Многочлены Чебышева** - множество многочленов $T_n(x)$, $n \ge 0$, определяемых соотношениями
$$T_0(x) = 1,\quad T_1(x) = x, \quad T_{n+1}(x) = 2xT_n(x) - T_{n-1}(x), \quad n=1,2,\dots.$$

$$T_n(x) = \cos(n\arccos{x}), \quad x \in [-1, 1], n \ge 0 \quad \text{- тригонометрическое представление многочленов Чебышева.}$$

$$\overline{T_n}(x) = 2^{1-n}T_n(x) \quad \text{- приведенный многочлен Чебышева.}$$

**Теорема.** Если $P_n(x)$ – это многочлен степени $n$ со старшим коэффициентом равным единице, то 
$$\max\limits_{x \in [-1,1]} \left| P_n(x) \right| \ge \max\limits_{x \in [-1,1]} \left| \overline{T_n}(x) \right|$$

Запишем многочлен Чебышева, минимально отклоняющийся от нуля на отрезке $[a, b]$:

$$T_{n+1}(x) = \frac{(b-a)^{n+1}}{2^{2n+1}}\cos\left((n+1)\arccos{\frac{2x-(b+a)}{b-a}}\right), \quad x \in [a,b]$$

Корни данного многочлена:

$$x_k = \frac{a+b}{2} + \frac{b-a}{2}\cos{\frac{(2k+1)\pi}{2(n+1)}}, \quad k = \overline{0,n}$$

Если выбрать узлами интерполирования $x_0, x_1, \dots, x_n$, совпадающие с корнями полинома, то величина отклонения $\omega_{n+1}(x)$ от нуля окажется минимальной. Кроме того, для упорядочивания узлов необходима перенумерация $\tilde{x}_k = x_{n-k}, \quad k = \overline{0,n}$.

Построим многочлен, используя новые узлы интерполирования, и рассчитаем невязку в точках восстановления:
$$\left|r(x^*)\right| = \left|P_{n}(x^*) - f (x^*)\right|$$

In [7]:
def chebyshev_nodes(a: float, b: float, n: int) -> list[float]:
    """Вывод n узлов Чебышёва на отрезке [a, b]
        a: начало отрезка
        b: конец отрезка
        n: количество узлов
    """
    return [float((a+b)/2 + (b-a)/2 *np.cos(((2*k+1)*np.pi)/(2*(n+1)))) for k in range(n+1)]

n = len(X) 

X_cheb = chebyshev_nodes(X[0], X[-1], n)
X_cheb.reverse()
F_x_cheb = [f(x) for x in X_cheb]
residuals = [abs(lagrange_polynomial(x, X_cheb, F_x_cheb) - f(x)) for x in recovery_x]

df = pd.DataFrame({
    "Точки восстановления": recovery_x,
    "f(x)": [f(x) for x in recovery_x],
    "P_n(x)": [lagrange_polynomial(x, X_cheb, F_x_cheb) for x in recovery_x],
    "Невязка": residuals
})
df = df.style.format({"Невязка": "{:.5e}"})
df

,Точки восстановления,f(x),P_n(x),Невязка
0,0.316667,1.055845,1.055845,4.44089e-16
1,0.800000,1.078915,1.078915,0.00000e+00
2,1.216667,1.104060,1.104060,2.22045e-16


Теперь оценим теоретическую погрешность:

$$\left| r_n(x) \right| \le \frac{M}{(n+1)!} \cdot \frac{(b-a)^{n+1}}{2^{2n+1}},$$

где $\left| f^{(n+1)}(x)\right| \le M, \quad x \in [a,b]. $

В нашем случае при $n=10 \quad$ $11$-я производная будет иметь вид:
$f^{(11)}(x) = 0.25 e^x + 0.75 \sin x$

In [8]:
from math import factorial

def f_11_derivative(x: float) -> float:
    return 0.25 * np.exp(x) + 0.75 * np.sin(x)

n = 10
x = np.linspace(X[0], X[-1], 1000)
M = np.max([abs(f_11_derivative(xi)) for xi in x])

r_theoretical = abs((M / factorial(n+1)) * ((X[-1] - X[0])**(n+1))/(2**(2*n+1)))
print("Теоретическая оценка остатка интерполирования в форме Ньютона:", r_theoretical)


Теоретическая оценка остатка интерполирования в форме Ньютона: 1.892598231951441e-14


Полученная практическая невязка интерполяции в точках восстановления имеет порядок $10^{-16}$, что существенно меньше теоретической оценки погрешности $1.89 \cdot 10^{-14}$. Это подтверждает корректность построения интерполяционного многочлена и согласуется с теоретической оценкой, которая задаёт лишь верхнюю границу ошибки.

---

### **6. Построение интерполяционного многочлена Ньютона внутри таблицы.**

Пусть функция задана в равноотстоящих узлах
$x_i = x_0 + ih, \quad h = const$.

Для интерполирования в точке x, расположенной вблизи середины между узлами $x_n$ и $x_{n+1}$, т.е $x \in [x_n, x_n + h]$ используется формула Ньютона–Бесселя. Вводится параметр:
$$t = \frac{x - x_n}{h}$$

В основе метода лежит аппарат конечных разностей:
$$\Delta f_i = f_{i+1} - f_i$$
$$\Delta^2 f_i = \Delta f_{i+1} - \Delta f_i$$
$$\Delta^k f_i = \Delta (\Delta^{k-1} f_i)= \Delta^{k-1} f_{i+1} - \Delta^{k-1} f_i$$

Тогда интерполяционный многочлен Ньютона-Бесселя будет выглядеть следующим образом:
$$P_{2k-1}(x) = P_{2k-1}(x + th) = \frac{f_n + f_{n+1}}{2} + \frac{t - \frac{1}{2}}{1!}\Delta f_n + \frac{t(t-1)}{2!} \cdot \frac{\Delta^2 f_n + \Delta^2 f_{n-1}}{2} + \frac{t(t-1)\left(t - \frac{1}{2}\right)}{3!} \Delta^3 f_{n-1} + \dots$$

Формула использует симметрично расположенные узлы:
$(x_n, x_{n+1}), (x_{n-1}, x_{n+2}), \dots$


Остаток интерполирования имеет вид:
$$r_{2k-1}(x) = \frac{h^{2k}}{(2k)!} \cdot t(t^2 - 1^2)(t^2 - 2^2)\dots(t^2 - (k-1)^2) \cdot f^{(2k)}(\xi), \quad \xi \in [x_n - kh + h, x_n + kh]$$

$$\left|r_{2k-1} (x)\right| \le \frac{h^{2k}}{(2k)!} \left|t(t^2 - 1^2) \dots(t^2 - (k-1)^2) \right|\max_{\xi \in [x_n - kh + h, x_n + kh]} \left|f^{(2k)}(\xi)\right|$$

Реализуем кубический интерполяционный многочлен ($k=2$) для точки восстановления $x^{**}$ из середины таблицы и рассчитаем практическую погрешность $\left|f(x^{**}) - P_n(x^{**})\right|$ и теоретическую, где

$$f^{(4)}(x) = 0.25 e^x + 0.75 \cos{x}$$

In [ ]:
def finite_difference_2d(F_x: list[float]) -> list[list[float]]:
    """
    Вычисление разделенной разности f(x_i, ..., x_j), используя двумерный массив
    F_x: список значений функции в узлах
    """
    n = len(F_x)
    dp = [[0.0] * n for _ in range(n)]
    dp[0] = F_x.copy()

    for i in range(1, n):
        for j in range(n - i):
            dp[i][j] = dp[i-1][j + 1] - dp[i-1][j]
    return dp

def newton_bessel_polynomial(x: float, X: list[float], F_x: list[float]) -> tuple[float, float, list[list[float]]]:
    """
    Вычисление интерполяционного многочлена в форме Ньютона-Бесселя третьего порядка для точки x
    x: точка, в которой нужно вычислить многочлен
    X: список равностоящих узлов
    F_x: список значений функции в узлах
    """

    def f_4_derivative(x: float) -> float:
        return 0.25 * np.exp(x) + 0.75 * np.cos(x)

    index = 0
    for i in range(len(X)-1):
        if X[i] <= x <= X[i+1]:
            x_n = X[i]
            index = i
            break
        
    k = 2
    h = X[1] - X[0]
    t = (x - x_n) / h

    table = finite_difference_2d(F_x)
    delta_0 = (table[0][index] + table[0][index+1])/2
    delta_1 = table[1][index]
    delta_2 = (table[2][index] + table[2][index-1])/2
    delta_3 = table[3][index-1]

    result = delta_0 + (t-0.5)*delta_1 + t*(t-1)*delta_2/factorial(2) + t*(t-1)*(t-0.5)*delta_3/(factorial(3))

    ksi = np.linspace(x_n - k*h + h, x_n + k*h, 1000)
    M_f = max(abs(f_4_derivative(ksi_i)) for ksi_i in ksi)

    residual = h**(2*k) * t*(t**2 -1) * M_f / factorial(2*k)
    return result, abs(residual), table

x_mid = recovery_x[1]
result, r_theoretical, table = newton_bessel_polynomial(x_mid, X, F_x)
r_practical = abs(result - f(x_mid))

df = pd.DataFrame({
    "Точка восстановления из середины": x_mid,
    "f(x)": f(x_mid),
    "P_n(x)": result,
    "Теоретическая невязка": r_theoretical,
    "|f(x) - P_n(x)|": r_practical
},index=[0])
df = df.style.format({"Теоретическая невязка": "{:.5e}", "|f(x) - P_n(x)|": "{:.5e}"})
display(df)

df_table = pd.DataFrame(table).T
n = len(F_x)

for i in range(n):
    df_table.iloc[i, n-i:] = np.nan

df_table.index = [f"x{i}" for i in range(n)]
df_table.columns = [f"Δ^{i}" for i in range(n)]
df_table.insert(0, "x", X)

df_table = df_table.style.format(
    {col: "{:.5e}" for col in df_table.columns if col != "x"},
    na_rep=""
)

print("\nТаблица конечных разностей:")
display(df_table)

1.0826897319268742


,Точка восстановления из середины,f(x),P_n(x),Теоретическая невязка,|f(x) - P_n(x)|
0,0.800000,1.078915,1.078913,1.69170e-06,2.52877e-06



Таблица конечных разностей:


,x,Δ^0,Δ^1,Δ^2,Δ^3,Δ^4,Δ^5,Δ^6,Δ^7,Δ^8,Δ^9,Δ^10
x0,0.250000,1.04769e+00,1.16058e-02,-3.48880e-03,6.65121e-04,1.06694e-04,5.42191e-07,-2.03399e-07,8.79108e-08,1.07556e-08,2.39522e-11,-1.47038e-12
x1,0.350000,1.05930e+00,8.11695e-03,-2.82368e-03,7.71816e-04,1.07236e-04,3.38792e-07,-1.15488e-07,9.86665e-08,1.07796e-08,2.24818e-11,
x2,0.450000,1.06741e+00,5.29327e-03,-2.05186e-03,8.79052e-04,1.07575e-04,2.23304e-07,-1.68217e-08,1.09446e-07,1.08020e-08,,
x3,0.550000,1.07271e+00,3.24141e-03,-1.17281e-03,9.86627e-04,1.07799e-04,2.06482e-07,9.26244e-08,1.20248e-07,,,
x4,0.650000,1.07595e+00,2.06860e-03,-1.86183e-04,1.09443e-03,1.08005e-04,2.99106e-07,2.12872e-07,,,,
x5,0.750000,1.07802e+00,1.88242e-03,9.08243e-04,1.20243e-03,1.08304e-04,5.11979e-07,,,,,
x6,0.850000,1.07990e+00,2.79066e-03,2.11067e-03,1.31074e-03,1.08816e-04,,,,,,
x7,0.950000,1.08269e+00,4.90133e-03,3.42141e-03,1.41955e-03,,,,,,,
x8,1.050000,1.08759e+00,8.32274e-03,4.84096e-03,,,,,,,,
x9,1.150000,1.09591e+00,1.31637e-02,,,,,,,,,


## **7. Вывод**

В ходе работы были исследованы различные методы аппроксимации и интерполирования функции: метод наименьших квадратов, интерполяция многочленом Лагранжа (на равномерной и чебышёвской сетках) и Ньютона–Бесселя.

| Метод              | Max ошибка  | Min ошибка  |
|:-------------------|------------:|------------:|
| МНК                | 4.60907e-08 | 8.32533e-09 |
| Многочлен Лагранжа     | 1.11910e-13 | 1.11022e-15 |
| Многочлен Лагранжа с чебышевской сеткой узлов | 4.44089e-16 | 0.00000e+00 |
| Интерполирование в середине таблицы    | 2.52877e-06 |-------------|

Наименьшую погрешность демонстрирует интерполяция многочленом Лагранжа на чебышёвской сетке — ошибки достигают минимума и лежат в пределах $\varepsilon \sim 10^{-16}$, что подтверждает её теоретическое преимущество как оптимального выбора узлов. Интерполяционный многочлен Лагранжа также показывает очень высокую точность $\varepsilon \sim 10^{-13} - 10^{-15}$, однако уступает чебышёвскому распределению узлов.

Метод наименьших квадратов даёт заметно большую погрешность $\varepsilon \sim 10^{-8} - 10^{-9}$, что ожидаемо, поскольку он не интерполирует функцию точно, а лишь аппроксимирует её, минимизируя суммарную квадратичную ошибку в среднем, а не в каждой точке. Кроме того, точность аппроксимации была ограничена предварительно заданной степенью многочлена (5-я степень), в то время как в интерполяционных методах степень определяется количеством узлом.

Интерполирование в середине таблицы при помощи формулы Ньютона–Бесселя имеет наибольшую ошибку $\varepsilon \sim 10^{-6}$, что связано с использованием многочлена 3-й степени, поэтому для интерполирования использовалось наименьшее количество узлов.

Во всех рассмотренных методах выполняется соотношение $|r_{\text{практ}}(x)| \leq |r_{\text{теор}}(x)|$ в пределах одного порядка, что подтверждает корректность реализации формул, правильность оценки и соответствие численных экспериментов теории.